In [1]:
# 0. Verisetini Hazırla
import pandas as pd
df = pd.read_csv("data/winequality_combined.csv")
df.head()

# Grover Dilemma: Type A'dan kaçınmak için 6500 satırlık verisetini kırpıyoruz.
# Sebep: log2(6500) yaklaşık= 13 Kübit. Olası indis durumları: 2^13 = 8192 !! 1692 SAHTE İNDİS DEMEK !!
# Arama sonucu olmayan indis sonuçlarını döneceği için 2^12 = 4098 GERÇEK İNDİS DEĞER'e eşledik böylece-
# Type A Grover Dilemma'sından kurtulmuş olduk.
df_ = df.copy().iloc[:8]

alcohol_ideal_decimal = 2
density_ideal_decimal = 4
olcek = {"alcohol": 10**alcohol_ideal_decimal, "density": 10**density_ideal_decimal}
print("Değişkenlerin en fazla alabildiği ondalık terim sayısı:")
for col in [col for col in df.columns if col not in ["alcohol", "density", "type"]]:
    max_decimal = df_[col].astype(str).str.split('.').str[1].fillna('').str.len().max()
    print("",col, max_decimal)
    df_[col] = (df[col].round(max_decimal) * (10 ** max_decimal)).round().astype(int)
    olcek[col] = 10 ** max_decimal 
## 'type' bağımlı değişken olduğu için 0 veya 1 değerleri atansa yeterli.
df_["type"] = (df_["type"]=="white").astype(int)
olcek["type"] = 1
print("Veriler hazır. Veriseti artık QROM + SAT boru hattı için hazır.")
df_

Değişkenlerin en fazla alabildiği ondalık terim sayısı:
 fixed acidity 1
 volatile acidity 2
 citric acid 2
 residual sugar 1
 chlorides 3
 free sulfur dioxide 1
 total sulfur dioxide 1
 pH 2
 sulphates 2
 quality 0
Veriler hazır. Veriseti artık QROM + SAT boru hattı için hazır.


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,74,70,0,19,76,110,340,0.9978,351,56,9.4,5,0
1,78,88,0,26,98,250,670,0.9968,320,68,9.8,5,0
2,78,76,4,23,92,150,540,0.9970,326,65,9.8,5,0
3,112,28,56,19,75,170,600,0.9980,316,58,9.8,6,0
4,74,70,0,19,76,110,340,0.9978,351,56,9.4,5,0
5,74,66,0,18,75,130,400,0.9978,351,56,9.4,5,0
6,79,60,6,16,69,150,590,0.9964,330,46,9.4,5,0
7,73,65,0,12,65,150,210,0.9946,339,47,10.0,7,0


In [2]:
df_[(df_["fixed acidity"]==73) & (df_["quality"]>=7)]

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
7,73,65,0,12,65,150,210,0.9946,339,47,10.0,7,0


In [3]:
# 1. Python + QDK entegrasyonu gerçek bir Azure Workspace'i ile uygulanmaya hazır hale getiriliyor.

# Varsayılan olarak 'rigetti.sim.qvm' devre simülatörü seçilmiştir. Dilerseniz farklı bir simülasyon-
# veya fiziksel Kuantum devreleri seçebilirsiniz. 
# Not: Azure ortamında çalışmak için Azure'a kayıt olunmalı, 'Quantum Workspace' sorgusu sonrası ilgili-
# adımları uygulayarak Kuantum Çalışma Alanı oluşturulmalı ve dilediğiniz devre API'ını planınıza dahil-
# edip işlemleri tamamlamalısınız. Sonra ilgili Çalışma Alanına gidip 'Resource ID' parametresini kopya-
# layıp '.env' dosyası içerisindeki 'RESOURCE_ID' alanına yapıştırın.

from qdk import qsharp
from azure.quantum import Workspace
from dotenv import load_dotenv
import os

load_dotenv()
qsharp.init(project_root=".", target_profile=qsharp.TargetProfile.Base)
resource_id = os.getenv("RESOURCE_ID")
print("Azure Çalışma Ortamı Kaynak Kimliği:",resource_id + "\n")
workspace = Workspace(resource_id=resource_id)
targets = workspace.get_targets()
print("### Mevcut Çalıştırma Hedefleri ###")
for i, t in enumerate(targets, 1):
    print(f"{i}. Hedef:\n İsim: {t.name}\n Ort. Gecikme(s->saniye): {t.average_queue_time}\n Durum:{'MEVCUT' if t.current_availability == 'Available' else 'KULLANIM DIŞI'}")
target_str = "quantinuum.sim.h2-1sc" #"rigetti.sim.qvm"

Azure Çalışma Ortamı Kaynak Kimliği: /subscriptions/f5f81187-0841-4323-b7c9-26f533275f3d/resourceGroups/AzureQuantum/providers/Microsoft.Quantum/Workspaces/quantum-ws-56391312

### Mevcut Çalıştırma Hedefleri ###
1. Hedef:
 İsim: rigetti.sim.qvm
 Ort. Gecikme(s->saniye): 5
 Durum:MEVCUT
2. Hedef:
 İsim: quantinuum.sim.h2-1sc
 Ort. Gecikme(s->saniye): 2
 Durum:MEVCUT
3. Hedef:
 İsim: quantinuum.sim.h2-1e
 Ort. Gecikme(s->saniye): 2350
 Durum:MEVCUT


In [4]:
# 2. Grover Algoritması içerisine gönderilecek olan değişken ve aritmetik işlemler Q#'a uygun hale getirmek için ara işlemler uyguluyoruz.
OPS = {"==": 0, ">": 1, "<": 2, ">=": 3, "<=": 4, "!=":5}
# kolon_sirasi = ["fixed acidity", "volatile acidity", "citric acid",
#                 "residual sugar", "chlorides", "free sulfur dioxide",
#                 "total sulfur dioxide", "pH", "sulphates",
#                 "alcohol", "density", "quality", "type"]
kolon_sirasi = ["fixed acidity", "quality"]


In [5]:
# 3. Her bir sütun için bit genişliği bilgilerini oluşturup saklamalıyız. Böylece Q#'ta belirli sorgu için yapılacak olan kübit işlemleri doğru şekilde
# uygulanmış olur

genislik = {}
print(f"Sütunların bit genişlikleri:")
for col in kolon_sirasi:
    max_val = int(df_[col].max())
    
    bit_length = max_val.bit_length()
    print(f" {col}: {bit_length}")
    genislik[col] = bit_length



def satir_to_bits(satir):
    tum_bitler = []

    for col in kolon_sirasi:
        deger = int(satir[col])
        
        bit_sayisi = genislik[col]

        ikili_string = format(deger, f"0{bit_sayisi}b")
        for karakter in ikili_string:
            if karakter == '1':
                tum_bitler.append(True)
            else:
                tum_bitler.append(False)

    return tum_bitler

dataset = df_[kolon_sirasi].apply(satir_to_bits, axis=1).tolist()

# Kontrol için ilk satırı yazdıralım:
print(dataset[0])
# çıktı: [False, False, True, True, ...] gibi uzun bir liste olacak

# Yapısal bilgiler
print(len(dataset))



Sütunların bit genişlikleri:
 fixed acidity: 7
 quality: 3
[True, False, False, True, False, True, False, True, False, True]
8


In [6]:
# 4. Q#'ın sütun değişkenlerini ayırt etmesi için bit-offset oluşturuyoruz.
offset = {}
konum = 0
print("Her bir sütunun bit-offset değeri:")
for col in kolon_sirasi:
    offset[col]=konum
    print(f" {col}: {konum}")
    konum+=genislik[col]

Her bir sütunun bit-offset değeri:
 fixed acidity: 0
 quality: 7


In [7]:
# 5. Q# içerisinde kullanmak için uyumlu bir betik sorgusu formatı oluşturuyoruz.
def q(col, op, deger):
    v = int(round(deger * olcek[col]))
    assert v.bit_length() <= genislik[col], f"{col}: {v} değeri {genislik[col]} bite sığmıyor"
    return (offset[col], genislik[col], OPS[op], v)

# (offset, genislik, operatör, değer)
queries = [
    q("fixed acidity", "==", 7.3),
    q("quality", ">=", 7)
]


In [ ]:
# 6. Q# devresine işlemler gönderilir ve hesaplama başlatılır.
# Dönecek olan sonuç
try:
    print(f"'{target_str}' seçiliyor...")
    target = workspace.get_targets(target_str)
except Exception as e:
    print(f"Bilinmeyen bir hata oluştu:\n{e}")
print("Seçim tamamlandı. Derleme işlemine geçildi.\n")
app_class = "Main.GroverSearchAlgorithm"
op = qsharp.eval(app_class)
print(f"'{app_class}' Sınıfı derleniyor...")
program = qsharp.compile(op, queries, dataset)
print(f"Derleme bitti. İş akışı {target_str}'e gönderildi.")
job = target.submit(program, "MicrosoftFY26GroverJob", shots=100)
print("İş akışı tamamlandı. Sonuç:")
try:
    results = job.get_results()
except:
    print("Timeout yedik. Limitsiz fallback uygulanıyor...")
    job.refresh()
    results = job.get_results(timeout_secs=None)   # ya da timeout_secs=7200

print(results)


'quantinuum.sim.h2-1sc' seçiliyor...
Seçim tamamlandı. Derleme işlemine geçildi.

'Main.GroverSearchAlgorithm' Sınıfı derleniyor...
Derleme bitti. İş akışı quantinuum.sim.h2-1sc'e gönderildi.
İş akışı tamamlandı. Sonuç:
....................

TimeoutError: The wait time has exceeded 300 seconds.

In [10]:
job.refresh()
results = job.get_results(timeout_secs=None)   # ya da timeout_secs=7200
print(results)

.....

KeyboardInterrupt: 

In [ ]:
sc = workspace.get_targets("quantinuum.sim.h2-1sc")
job_sc = sc.submit(program, "grover-syntax-check", shots=10)
print(job_sc.get_results())   # tasarım gereği hep 0 döner — önemli olan hatasız 'Succeeded' görmek

In [ ]:
mini = qsharp.compile("{ use qs = Qubit[3]; CCNOT(qs[0], qs[1], qs[2]); MResetEachZ(qs) }")
job_mini = sc.submit(mini, "mini-sc-testi", shots=10)
print(job_mini.get_results())

In [ ]:
# 7. Sonuçları gözlemle.
from collections import Counter

def decode(shot):  # little-endian: ilk eleman LSB
    return sum((1 if str(b) == "One" else 0) << i for i, b in enumerate(shot))

# yerel doğrulama — Azure'a hiç gitmeden
# app_class = "Main.GroverSearchAlgorithm"
# op = qsharp.eval(app_class)
# sonuclar = qsharp.run(op, 100, queries, dataset)
# hist = Counter(decode(s) for s in sonuclar)
hist = Counter(decode(s) for s in results)
print(hist.most_common(5))

In [ ]:
mask = (df_["fixed acidity"] == 73) & (df_["quality"] >= 7)
print(df_[mask].index.tolist())   # beklenen: [7]

In [ ]:
## HATA AYIKLAMA ##

qir = str(program)
with open("program.ll", "w") as f: f.write(qir)

import re
from collections import Counter
print(Counter(re.findall(r"__quantum__qis__(\w+?)__", qir)))
print(re.findall(r'required_num_(qubits|results)"="(\d+)', qir))

mini = qsharp.compile("{ use qs = Qubit[3]; CCNOT(qs[0], qs[1], qs[2]); MResetEachZ(qs) }")
job2 = target.submit(mini, "ccx-testi", shots=10)
job2.get_results()